In [1]:
# 1. SETUP & IMPORTS
!pip install -q diffusers transformers accelerate gTTS moviepy
import os, torch, numpy as np
from diffusers import AutoPipelineForText2Image
from PIL import Image, ImageDraw, ImageFont
from moviepy.editor import ImageClip, AudioFileClip, concatenate_videoclips, ColorClip
from gtts import gTTS
from google.colab import files

# 2. CONFIGURATION
FOLDER_PATH = "/content/Ramayana_Project"
os.makedirs(f"{FOLDER_PATH}/panels", exist_ok=True)
os.makedirs(f"{FOLDER_PATH}/audio", exist_ok=True)

# Initialize Model (SDXL Turbo)
pipe = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sdxl-turbo", torch_dtype=torch.float16, variant="fp16"
).to("cuda")

# Helper: Drawing "Face-Safe" Subtitles
def add_subtitles(image, text):
    canvas = image.copy().convert("RGBA")
    draw = ImageDraw.Draw(canvas)
    width, height = canvas.size
    bar_h = 80
    draw.rectangle([0, height-bar_h, width, height], fill=(0, 0, 0, 160)) # Semi-transparent
    draw.text((20, height - 55), text, fill="white")
    return canvas.convert("RGB")

print("⚙️ Engine Ready: Directories and Model loaded.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 11.1 MB/s eta 0:00:00


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning

model_index.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/565 [00:00<?, ?B/s]

...

unet/diffusion_pytorch_model.fp16.safete(…):   0%|          | 0.00/5.14G [00:00<?, ?B/s]

vae/diffusion_pytorch_model.fp16.safeten(…):   0%|          | 0.00/167M [00:00<?, ?B/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!


⚙️ Engine Ready: Directories and Model loaded.


In [3]:
# --- STEP 1: DEFINE THE STORY DATA ---
chapters = [
    {
        "title": "Chapter 1: Balakanda",
        "intro": "In the golden city of Ayodhya, a hero is born.",
        "prompt": "Young Prince Rama with blue skin and a gold crown, playing in the palace garden, simple rounded shapes",
        "dialogue": "I will always follow the path of truth.",
        "filename": "chap1"
    },
    {
        "title": "Chapter 2: Ayodhya Kanda",
        "intro": "Duty calls Rama to leave his home for the deep forest.",
        "prompt": "Rama and Sita walking into a peaceful forest, simple sage clothes, brave faces",
        "dialogue": "Do not worry. We will return soon.",
        "filename": "chap2"
    },
    {
        "title": "Chapter 3: Aranya Kanda",
        "intro": "In the wild forest, a mysterious golden deer appears.",
        "prompt": "A magical golden deer with glowing eyes in a colorful jungle",
        "dialogue": "Rama, look! The most beautiful deer in the world!",
        "filename": "chap3"
    },
    {
        "title": "Chapter 4: Kishkindha Kanda",
        "intro": "Rama meets the brave monkey king Sugriva.",
        "prompt": "Rama shaking hands with the monkey king Sugriva, green valley",
        "dialogue": "Together, we will find Sita!",
        "filename": "chap4"
    },
    {
        "title": "Chapter 5: Sundara Kanda",
        "intro": "The mighty Hanuman leaps across the ocean.",
        "prompt": "Hanuman flying over a blue ocean with a mountain in his hand",
        "dialogue": "Jai Shri Ram! I have found the Princess!",
        "filename": "chap5"
    },
    {
        "title": "Chapter 6: Yuddha Kanda",
        "intro": "The final battle between good and evil begins.",
        "prompt": "Rama aiming a golden arrow at the 10-headed king Ravana",
        "dialogue": "The time for darkness is over!",
        "filename": "chap6"
    },
    {
        "title": "Chapter 7: Uttara Kanda",
        "intro": "Victory is won, and Rama returns home.",
        "prompt": "Rama and Sita on a gold throne with glowing oil lamps",
        "dialogue": "Happiness has returned. Diwali is here!",
        "filename": "chap7"
    }
]

print("✅ Chapters defined and ready for the Producer!")

✅ Chapters defined and ready for the Producer!


In [4]:
def produce_epic_movie(script):
    all_scenes = []

    for i, chap in enumerate(script):
        print(f"🎬 Processing Chapter {i+1}: {chap['title']}")

        # A. Generate Image
        style = ", modern children's book illustration, flat vector art, clean lines, white background elements"
        img = pipe(prompt=chap['prompt'] + style, num_inference_steps=2, guidance_scale=0.0).images[0]
        img_sub = add_subtitles(img, chap['dialogue'])

        # B. Generate Voiceover
        voice_path = f"{FOLDER_PATH}/audio/v_{i}.mp3"
        full_text = f"{chap['intro']}. {chap['dialogue']}"
        gTTS(full_text, lang='en').save(voice_path)

        # C. Create Clip (Auto-timed to voice)
        audio = AudioFileClip(voice_path)
        clip = ImageClip(np.array(img_sub)).set_duration(audio.duration + 0.5).set_audio(audio)
        all_scenes.append(clip)

    # D. Final Assembly
    print("🎥 Finalizing Movie...")
    final_video = concatenate_videoclips(all_scenes, method="compose")

    # Add 'The End' Card
    end_card = ColorClip(size=(512,512), color=(0,0,0), duration=3)
    final_master = concatenate_videoclips([final_video, end_card], method="compose")

    final_master.write_videofile(f"{FOLDER_PATH}/Ramayana_Final.mp4", fps=24, codec="libx264")
    files.download(f"{FOLDER_PATH}/Ramayana_Final.mp4")

# RUN IT!
produce_epic_movie(chapters)

🎬 Processing Chapter 1: Chapter 1: Balakanda


  0%|          | 0/2 [00:00<?, ?it/s]

  deprecate(



🎬 Processing Chapter 2: Chapter 2: Ayodhya Kanda


  0%|          | 0/2 [00:00<?, ?it/s]

...

🎥 Finalizing Movie...
Moviepy - Building video /content/Ramayana_Project/Ramayana_Final.mp4.
MoviePy - Writing audio in Ramayana_FinalTEMP_MPY_wvf_snd.mp3


MoviePy - Done.
Moviepy - Writing video /content/Ramayana_Project/Ramayana_Final.mp4



Moviepy - Done !
Moviepy - video ready /content/Ramayana_Project/Ramayana_Final.mp4


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>